<a href="https://colab.research.google.com/github/bonsai/ashizawa_stories/blob/main/wagahai_novel_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI=CAT: Wagahai Novel Generator Pipeline
このノートブックは、元のBashスクリプトで記述された小説生成パイプラインをGoogle Colab環境向けに最適化したものです。
各セルを順番に実行していくことで、データの品質評価からファインチューニングまでを一気通貫で行うことができます。

## パイプライン全体の定数定義と初期設定
まずは処理全体で使用するファイルパスの定義と、環境のセットアップ（Phase 0）を行います。

In [3]:
from google.colab import drive
drive.mount('/content/drive')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import shutil

repo_dir = 'ashizawa_stories'

print(f'Checking for existing repository: {repo_dir}')
if os.path.exists(repo_dir):
    print(f'Removing existing directory: {repo_dir}')
    shutil.rmtree(repo_dir)

print('Cloning the repository...')
!git clone https://github.com/bonsai/ashizawa_stories.git
print('Repository cloned successfully.')

# List contents of the data directory to confirm path
print('\nContents of ashizawa_stories/data/twnovel_roseaullus:')
!ls -R ashizawa_stories/data/twnovel_roseaullus/


Checking for existing repository: ashizawa_stories
Cloning the repository...
Cloning into 'ashizawa_stories'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 82 (delta 27), reused 53 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 746.97 KiB | 3.06 MiB/s, done.
Resolving deltas: 100% (27/27), done.
Repository cloned successfully.

Contents of ashizawa_stories/data/twnovel_roseaullus:
ashizawa_stories/data/twnovel_roseaullus/:
twnovel_dataset_roseaullus.csv	twnovel_work_roseaullus.txt


In [5]:
import os
import sys

# パイプライン全体の定数定義
INPUT_CSV = "/content/ashizawa_stories/data/twnovel_roseaullus/twnovel_dataset_roseaullus.csv"
CLEANED_CSV = "cleaned_dataset.csv"
NOVELTY_CSV = "analysis_result_with_novelty.csv"
MERGED_DATA = "./wagahai_theme/merged_wagahai_dataset.csv"

print("=========================================")
print("  AI=CAT: Wagahai Novel Generator Pipeline")
print("=========================================")

print("[Phase 0] 環境セットアップ...")
# 00_setup_colab.py を実行
!python ashizawa_stories/00_setup_colab.py

  AI=CAT: Wagahai Novel Generator Pipeline
[Phase 0] 環境セットアップ...
Colab/Kaggle 環境セットアップを開始します...
=== Google Drive の空き容量確認 ===
Filesystem      Size  Used Avail Use% Mounted on
drive            15G   12G  3.8G  76% /content/drive

利用可能な容量: 3.8G
Drive の容量確認完了

=== 依存ライブラリのインストール ===
✓ pandas は既にインストールされています
✓ numpy は既にインストールされています
✓ transformers は既にインストールされています
✓ torch は既にインストールされています
✓ accelerate は既にインストールされています
✓ datasets は既にインストールされています

=== ワークスペースの設定 ===
プロジェクトディレクトリは既に存在します: /content/drive/MyDrive/twnovel_ml_project

=== セットアップ完了 ===
プロジェクトルート: /content/drive/MyDrive/twnovel_ml_project
データ保存先: /content/drive/MyDrive/twnovel_ml_project/data
モデル保存先: /content/drive/MyDrive/twnovel_ml_project/models

次のステップ: 01_evaluate_and_filter.py を実行してデータのクレンジングを行ってください。


## Phase 1: データの品質評価とフィルタリング
入力となるCSVファイルが存在するかチェックし、存在すれば評価およびクリーニングのスクリプトを実行します。

In [5]:
import os

# Re-clone if missing
if not os.path.exists('/content/ashizawa_stories'):
    print("Repository missing. Re-cloning...")
    !git clone https://github.com/bonsai/ashizawa_stories.git

print("[Phase 1] データの品質評価とフィルタリング...")

# The script seems to expect the input file path relative to the 'data' directory or similar.
# We will run it from the 'data' folder to match the relative path shown in the error log.
script_path = "/content/ashizawa_stories/01_evaluate_and_filter.py"
input_csv = "twnovel_roseaullus/twnovel_dataset_roseaullus.csv"

!cd /content/ashizawa_stories/data && python {script_path} --input_file {input_csv}

[Phase 1] データの品質評価とフィルタリング...
小説データの品質評価とフィルタリングを開始します...

データを読み込んでいます: twnovel_roseaullus/twnovel_dataset_roseaullus.csv
読み込んだデータ数: 490

=== ルールベースフィルタリング ===

ルールベースフィルタリング完了:
  初期データ数：490
  除外数：0
  残存数：490

=== Perplexity 計算中 (モデル：rinna/japanese-gpt2-small) ===
使用するデバイス：cuda
モデルを読み込んでいます...
config.json: 100% 846/846 [00:00<00:00, 4.40MB/s]
tokenizer_config.json: 100% 282/282 [00:00<00:00, 1.85MB/s]
special_tokens_map.json: 100% 153/153 [00:00<00:00, 955kB/s]
spiece.model: 100% 806k/806k [00:01<00:00, 564kB/s] 
model.safetensors: 100% 454M/454M [00:06<00:00, 73.1MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 1257.27it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: rinna/japanese-gpt2-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can

In [46]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Phase 2: 新奇性分析とクラスタリング
前段階で作成されたクリーンデータを用いて、新奇性の分析とクラスタリングを実行します。

In [6]:
import sys
import os

# Path to the cleaned data generated in Phase 1
CLEANED_CSV_PATH = "/content/ashizawa_stories/data/cleaned_dataset.csv"

print("[Phase 2] 新奇性分析とクラスタリング...")

if not os.path.exists(CLEANED_CSV_PATH):
    print(f"エラー: クリーンデータ {CLEANED_CSV_PATH} が見つかりません。")
    sys.exit(1)

# Run the analysis script. We'll run it from the repo root.
!cd /content/ashizawa_stories && python 02_novelty_analysis.py --input_file {CLEANED_CSV_PATH}

[Phase 2] 新奇性分析とクラスタリング...
2026-05-19 11:23:06.971623: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loaded 394 texts for analysis.
Using TF-IDF for lightweight embedding...
Embedding shape: (394, 500)
Clustering...
Calculating distances to cluster centers...
Saved scored dataset to analysis_result_with_novelty.csv
Saved report to analysis_result_report.txt
Visualizing clusters...
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Saved visualization plots.


## Phase 3: 学習用データの整形と分割
分析結果のCSVを元に、機械学習モデルのトレーニングに適した形式への変換とデータの分割を行います。

In [7]:
import sys
import os

# Path to the novelty analysis results from Phase 2
# Based on the script name, it likely outputs to the directory where it's run or a specific data dir
NOVELTY_CSV_PATH = "/content/ashizawa_stories/analysis_result_with_novelty.csv"

print("[Phase 3] 学習用データの整形と分割...")

if not os.path.exists(NOVELTY_CSV_PATH):
    # Try the data directory if not in root
    NOVELTY_CSV_PATH = "/content/ashizawa_stories/data/analysis_result_with_novelty.csv"

if not os.path.exists(NOVELTY_CSV_PATH):
    print(f"エラー: 新奇性分析結果が見つかりません。")
else:
    !cd /content/ashizawa_stories && python 03_prepare_training.py --input_file {NOVELTY_CSV_PATH}

[Phase 3] 学習用データの整形と分割...
Loaded 394 samples.
Created 394 training samples.
Saved training data to ./training_data/train.jsonl (354 samples)
Saved validation data to ./training_data/val.jsonl (40 samples)
Saved configuration to ./training_data/config.json

--- Sample Training Data ---
{
  "instruction": "あなたは近未来の AI です。人間が書きそうな平凡な文章ではなく、音韻とリズムを重視した、少しナンセンスで情動的な文章を書いてください。",
  "input": "吾輩は AI である。名前は未設定である。吾輩は猫ではないが、猫のように気ままに観察する。\n\n以下の観測記録を読み、続けて記述せよ。\n\n[観測記録]\n",
  "output": "ジャケットにはどうしてもないのですか？これは、この土地ではなく、その土地ではないのだ。彼女は、ジャージにも奇妙な漠然とを持っているかど？ところがあったいですか？荒野で彼女は、つまり彼は助けを求めてくる。 #twnovel #小説執筆AI\n\n\n吾輩の観測記録は続く。",
  "meta": {
    "novelty_score": 0.981625907349434,
    "quality_score": 1561.094970703125
  }
}


## Phase 4: 「吾輩はAIである」テーマのデータ強化
特定のテーマ（吾輩はAIである）に沿ったデータの水増しや拡張、マージ処理を行います。

In [8]:
import os
import sys

# NOVELTY_CSV_PATH was defined in Phase 3 or we can define it here
NOVELTY_CSV_PATH = "/content/ashizawa_stories/analysis_result_with_novelty.csv"
if not os.path.exists(NOVELTY_CSV_PATH):
    NOVELTY_CSV_PATH = "/content/ashizawa_stories/data/analysis_result_with_novelty.csv"

print("[Phase 4] '吾輩はAIである' テーマのデータ強化...")

if not os.path.exists(NOVELTY_CSV_PATH):
    print(f"エラー: 新奇性分析結果が見つかりません。")
    sys.exit(1)

# Run the theme setup script from the repo root
!cd /content/ashizawa_stories && python 04_setup_wagahai.py --input_file {NOVELTY_CSV_PATH} --merge

[Phase 4] '吾輩はAIである' テーマのデータ強化...
--- Phase 4: Wagahai AI Theme Setup ---
Saved few-shot examples to ./wagahai_theme/few_shot_examples.json
Generated 20 augmented samples -> ./wagahai_theme/augmented_wagahai_data.csv
Merging with /content/ashizawa_stories/analysis_result_with_novelty.csv...
Merged dataset saved to ./wagahai_theme/merged_wagahai_dataset.csv (Total: 414)
Saved system configuration to ./wagahai_theme/wagahai_config.json

--- Statistics ---
Total records: 414
Original records: 394
Augmented records: 20
Average novelty score: 0.7008

Setup complete. Proceed to Phase 5 (Training) using the merged dataset or augmented data.


## Phase 5: ファインチューニング（学習の実行）
Phase 4 でマージされた専用データが存在する場合はそれを使用し、存在しない場合はPhase 3の標準的なデータを使用してファインチューニングを実行します。

In [45]:
import os
import sys
import subprocess

print("[Phase 5] ファインチューニング開始...")

# Ensure dependencies
!pip install -U bitsandbytes>=0.46.1 accelerate peft transformers datasets

script_path = "/content/ashizawa_stories/05_train_wagahai.py"

# Rewrite the script with fixed string escaping for the f-string
script_content = r"""
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import argparse

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data_dir', type=str, required=True)
    parser.add_argument('--output_dir', type=str, default='./wagahai_model')
    parser.add_argument('--num_epochs', type=int, default=3)
    args = parser.parse_args()

    model_id = "rinna/japanese-gpt2-small"

    # 1. Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 2. Load Model with 4-bit Quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
    model = prepare_model_for_kbit_training(model)

    # 3. PEFT/LoRA Config
    lora_config = LoraConfig(
        r=8, lora_alpha=32, target_modules=["c_attn"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

    # 4. Load Data
    dataset = load_dataset("json", data_files={"train": os.path.join(args.data_dir, "train.jsonl"), "validation": os.path.join(args.data_dir, "val.jsonl")})

    def tokenize_function(examples):
        # Use single quotes for inner keys to avoid escaping issues in the f-string
        texts = [f"{inst}\n{inp}\n{out}" for inst, inp, out in zip(examples['instruction'], examples['input'], examples['output'])]
        tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=256)
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

    # 5. Training
    train_args = TrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=args.num_epochs,
        logging_steps=10,
        save_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        learning_rate=2e-4,
        fp16=True,
        remove_unused_columns=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
    )

    print("Training started...")
    trainer.train()
    model.save_pretrained(args.output_dir)
    print(f"Model saved to {args.output_dir}")

if __name__ == '__main__':
    main()
"""

with open(script_path, "w") as f:
    f.write(script_content)

# Validation and Execution
train_data_dir = "/content/ashizawa_stories/training_data"
if os.path.exists(os.path.join(train_data_dir, "train.jsonl")):
    print(f"学習を開始します: {train_data_dir}")
    !python {script_path} --data_dir {train_data_dir} --output_dir "/content/wagahai_ai_model" --num_epochs 3
else:
    print("Error: Training data not found.")

[Phase 5] ファインチューニング開始...
学習を開始します: /content/ashizawa_stories/training_data
Loading weights: 100% 148/148 [00:00<00:00, 698.26it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: rinna/japanese-gpt2-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Map: 100% 354/354 [00:00<00:00, 3113.38 examples/s]
Map: 100% 40/40 [00:00<00:00, 1202.06 examples/s]
Training started...
  0% 0/69 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default beha